In [1]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


pandas: 3.0.2  polars: 1.39.3


In [2]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- target_elusive_sample_pairs ---
def FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_FIND_PAIRS(group):
    samples = sorted(group["found_in"].to_list() if hasattr(group["found_in"], "to_list") else group["found_in"].tolist())
    return ["|".join(samples[:2])] if len(samples) >= 2 else []

def FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_GET_TAXA_GROUP(taxonomy):
    return "Bacteria" if "Bacteria" in str(taxonomy) else None

FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_GROUP = "group_A"
FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_TAXONOMY = pd.DataFrame({"taxonomy": ["Bacteria", "Archaea"]}, index=pd.Index(["0", "1"], name="target"))
FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_TAXONOMY_PL = pl.DataFrame({"target": ["0", "1"], "taxonomy": ["Bacteria", "Archaea"]})
FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_UNBINNED = pd.DataFrame({"target": ["0", "0", "1"], "found_in": ["s1", "s2", "s3"], "taxonomy": ["Bacteria", "Bacteria", "Archaea"]})
FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_UNBINNED_PL = pl.from_pandas(FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_UNBINNED)

# --- target_elusive_sparse_edges ---
FIX_TARGET_ELUSIVE_SPARSE_EDGES_SAMPLE_PAIRS = pd.DataFrame({"sample_pairs": ["s1|s2", "s1|s2", "s2|s3"], "target": ["0", "2", "1"], "taxa_group": ["Bacteria", "Bacteria", "Archaea"]})
FIX_TARGET_ELUSIVE_SPARSE_EDGES_SAMPLE_PAIRS_PL = pl.from_pandas(FIX_TARGET_ELUSIVE_SPARSE_EDGES_SAMPLE_PAIRS)

# --- target_elusive_unbinned ---
FIX_TARGET_ELUSIVE_UNBINNED_UNBINNED = pd.DataFrame({"found_in": ["s1", "s2", "s3"], "gene": ["g1", "g2", "g1"], "sequence": ["seq1", "seq2", "seq1"], "taxonomy": ["tax1", "tax2", "tax1"]})
FIX_TARGET_ELUSIVE_UNBINNED_UNBINNED_PL = pl.from_pandas(FIX_TARGET_ELUSIVE_UNBINNED_UNBINNED)

print("✅ Fixtures loaded")


✅ Fixtures loaded


In [3]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_target_elusive_sample_pairs(find_pairs, get_taxa_group, group, taxonomy, unbinned):
    sample_pairs = (unbinned
        .groupby("target")
        .apply(find_pairs)
        .to_frame("sample_pairs")
        .explode("sample_pairs")
        .dropna(subset = "sample_pairs")
        .join(taxonomy)
        .reset_index()
        )
    sample_pairs["taxa_group"] = sample_pairs["taxonomy"].apply(get_taxa_group)
    sample_pairs = sample_pairs.dropna(subset = "taxa_group")
    return sample_pairs

def before_target_elusive_sparse_edges(sample_pairs):
    sparse_edges = (sample_pairs
        .groupby(["taxa_group", "sample_pairs"])["target"]
        .agg(["count", lambda x: ",".join(sorted(x))])
        .reset_index()
        )
    sparse_edges.rename(columns = {"count": "weight", "<lambda_0>": "target_ids"}, inplace=True)
    return sparse_edges

def before_target_elusive_unbinned(unbinned):
    unbinned.drop(["found_in"], axis=1, errors="ignore", inplace=True)
    unbinned["target"] = unbinned.groupby(["gene", "sequence"]).ngroup()
    unbinned["target"] = unbinned["target"].astype(str)

    taxonomy = unbinned.groupby("target")["taxonomy"].first()
    return taxonomy

In [4]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_target_elusive_sample_pairs(find_pairs, get_taxa_group, group, taxonomy, unbinned):

    _sample_pairs_rows = []
    for target, data in unbinned.group_by("target", maintain_order=True):
        res = find_pairs(data)
        if isinstance(res, pl.DataFrame):
            values = res.to_series(0).to_list() if res.width == 1 else res.rows()
        elif isinstance(res, pl.Series):
            values = res.to_list()
        else:
            values = res
        if not isinstance(values, list):
            values = [values]
        for v in values:
            _sample_pairs_rows.append({"target": target[0] if isinstance(target, tuple) and len(target) == 1 else target, "sample_pairs": v})

    sample_pairs = pl.DataFrame(_sample_pairs_rows).explode("sample_pairs").drop_nulls(subset=["sample_pairs"])
    sample_pairs = pl.concat([sample_pairs, taxonomy], how="horizontal")
    sample_pairs = sample_pairs.with_columns(
        pl.col("taxonomy").map_elements(get_taxa_group, return_dtype=pl.Null).alias("taxa_group")
    )
    sample_pairs = sample_pairs.drop_nulls(subset=["taxa_group"])
    return sample_pairs

def gen_target_elusive_sparse_edges(sample_pairs):

    sparse_edges = (
        sample_pairs.group_by(["taxa_group", "sample_pairs"], maintain_order=False)
        .agg(
            pl.col("target").count().alias("count"),
            pl.col("target").sort().list.join(",").alias("<lambda_0>"),
        )
        .sort(["taxa_group", "sample_pairs"])
    )

    sparse_edges = sparse_edges.rename({"count": "weight", "<lambda_0>": "target_ids"})
    return sparse_edges

def gen_target_elusive_unbinned(unbinned):

    unbinned = unbinned.drop("found_in", strict=False)
    groups = (
        unbinned.select(["gene", "sequence"])
        .unique(maintain_order=False)
        .sort(["gene", "sequence"])
        .with_row_index("target")
        .with_columns(pl.col("target").cast(pl.Utf8))
    )
    unbinned = unbinned.join(groups, on=["gene", "sequence"], how="left")
    taxonomy = unbinned.group_by("target").agg(pl.col("taxonomy").first()).get_column("taxonomy")
    return taxonomy

In [5]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None


def _normalize_compare_frame(df):
    for name, dtype in zip(df.columns, df.dtypes):
        if dtype == pl.Object:
            df = df.with_columns(pl.col(name).map_elements(lambda x: None if x is None else str(x), return_dtype=pl.Utf8).alias(name))
    return df

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    left = _normalize_compare_frame(left)
    right = _normalize_compare_frame(right)
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [6]:
# === Tests: target_elusive_sample_pairs ===

try:
    _r = gen_target_elusive_sample_pairs(FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_FIND_PAIRS, FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_GET_TAXA_GROUP, FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_GROUP, FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_TAXONOMY_PL, FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_UNBINNED_PL)
    print("✅ L1 smoke gen_target_elusive_sample_pairs: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_target_elusive_sample_pairs: {type(_e).__name__}: {_e}")

try:
    _rb = before_target_elusive_sample_pairs(FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_FIND_PAIRS, FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_GET_TAXA_GROUP, FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_GROUP, FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_TAXONOMY, FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_UNBINNED)
    print("✅ L1 smoke before_target_elusive_sample_pairs: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_target_elusive_sample_pairs: {type(_e).__name__}: {_e}")

try:
    _rb = before_target_elusive_sample_pairs(FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_FIND_PAIRS, FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_GET_TAXA_GROUP, FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_GROUP, FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_TAXONOMY, FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_UNBINNED)
    _rg = gen_target_elusive_sample_pairs(FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_FIND_PAIRS, FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_GET_TAXA_GROUP, FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_GROUP, FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_TAXONOMY_PL, FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_UNBINNED_PL)
    compare(_rb, _rg, "target_elusive_sample_pairs")
except Exception as _e:
    print(f"❌ L2 equivalence target_elusive_sample_pairs: setup error — {type(_e).__name__}: {_e}")

# L3 edge – one-sample groups and non-matching taxonomy should be dropped.
try:
    _edge_pd = pd.DataFrame({"target": ["0", "1"], "found_in": ["s1", "s2"], "taxonomy": ["Bacteria", "Archaea"]})
    _edge_pl = pl.from_pandas(_edge_pd)
    _rb = before_target_elusive_sample_pairs(
        FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_FIND_PAIRS,
        FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_GET_TAXA_GROUP,
        FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_GROUP,
        FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_TAXONOMY,
        _edge_pd,
    )
    _rg = gen_target_elusive_sample_pairs(
        FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_FIND_PAIRS,
        FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_GET_TAXA_GROUP,
        FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_GROUP,
        FIX_TARGET_ELUSIVE_SAMPLE_PAIRS_TAXONOMY_PL,
        _edge_pl,
    )
    compare(_rb, _rg, "L3 edge target_elusive_sample_pairs singletons dropped")
except Exception as _e:
    print(f"❌ L3 edge target_elusive_sample_pairs: {type(_e).__name__}: {_e}")


❌ L1 smoke gen_target_elusive_sample_pairs: InvalidOperationError: `explode` operation not supported for dtype `str`
✅ L1 smoke before_target_elusive_sample_pairs: OK
❌ L2 equivalence target_elusive_sample_pairs: setup error — InvalidOperationError: `explode` operation not supported for dtype `str`
❌ L3 edge target_elusive_sample_pairs: ColumnNotFoundError: "sample_pairs" not found
